# How Much Accuracy Do You Need?

Original QMCPy demo: [`QMCPy/demos/demo_resume_data/accuracy_and_resume.ipynb`](../../../QMCPy/demos/demo_resume_data/accuracy_and_resume.ipynb)

Sou-Cheng Choi (with some edits by Fred Hickernell)

This Julia translation follows the same theme: start with a loose tolerance, then tighten the target only if you need more accuracy.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/demo_resume_data/accuracy_and_resume.ipynb)

## The Problem

Automatic quadrature routines require the user to specify a target accuracy. In practice, scientists often do not know that accuracy in advance, or their needs change after they inspect a first result.


## The Solution: Resumable Integration in QMC.jl

With resumable integration, the result of an earlier run is not wasted. A second solve at a tighter tolerance can continue from the saved state instead of restarting from zero.


## Implementation

The same implementation ideas used in QMCPy are available in `QMC.jl`: the `integrate` method accepts a `resume` argument, and the stopping criteria can record an iteration log when `trace_iterations=true`.


In [1]:
using QMC
using Serialization
using Printf


### Step 1: Quick Estimate

Suppose you want a quick answer, so you set a loose tolerance. We use a Genz oscillatory benchmark and a fixed seed to keep the notebook reproducible.


In [2]:
function make_solver(abs_tol; seed=7, dimension=3)
    dd = Lattice(dimension; seed=seed)
    f = Genz(Uniform(dd); kind=:oscillatory, a=ones(dimension), u=0.5 .* ones(dimension))
    return CubQMCLatticeG(f; abs_tol=abs_tol, trace_iterations=true)
end

loose_tol = 1e-3
result_loose = integrate(make_solver(loose_tol))
@printf("Loose solve: solution = %.8f, n_total = %d, time = %.4f s
", result_loose.solution, result_loose.data[:n_total], result_loose.data[:time_integrate])
sort!(collect(keys(result_loose.data)))


Loose solve: solution = -0.06222385, n_total = 16384, time = 0.0030 s


9-element Vector{Symbol}:
 :converged
 :error_bound
 :iteration_log
 :n
 :n_iterations
 :n_per_rep
 :n_reps
 :n_total
 :time_integrate

The returned data object contains the estimated solution, the error bound, the number of samples used, the iteration log, and the measured solve time.


### Step 2: Save the State (Optional)

You can save the integration state to disk and resume later, or keep the state in memory and continue immediately.


In [3]:
output_dir = joinpath(pwd(), "output")
mkpath(output_dir)
resume_path = joinpath(output_dir, "accuracy_and_resume_loose.jls")
serialize(resume_path, result_loose.data)
println("Saved state to: ", resume_path)


Saved state to: /Users/terrya/Documents/ProgramData/QMCSoftware_space/QMC.jl/demos/demo_resume_data/output/accuracy_and_resume_loose.jls

### Step 3: Resume with a Tighter Tolerance

Now assume you decide that the first answer is not accurate enough. Resume from the saved state and ask for a tighter tolerance.


In [4]:
tight_tol = 2.5e-4
result_resume = integrate(make_solver(tight_tol); resume=deserialize(resume_path))
@printf("Resumed solve: solution = %.8f, n_total = %d, incremental time = %.4f s
", result_resume.solution, result_resume.data[:n_total], result_resume.data[:time_integrate])


Resumed solve: solution = -0.06236903, n_total = 32768, incremental time = 0.0068 s


### Step 4: Compare with a Fresh Tight Solve

To see what the resume feature saved, solve the same tight problem again from scratch.


In [5]:
result_fresh = integrate(make_solver(tight_tol))
@printf("Fresh tight solve: solution = %.8f, n_total = %d, time = %.4f s
", result_fresh.solution, result_fresh.data[:n_total], result_fresh.data[:time_integrate])
@printf("Saved samples: %d
", result_fresh.data[:n_total] - result_loose.data[:n_total])
@printf("Resumed vs fresh difference: %.3e
", abs(result_resume.solution - result_fresh.solution))

@assert abs(result_resume.solution - result_fresh.solution) ≤ max(result_resume.data[:error_bound], result_fresh.data[:error_bound])
@assert result_resume.data[:n_total] == result_fresh.data[:n_total]


Fresh tight solve: solution = -0.06222642, n_total = 32768, time = 0.0009 s
Saved samples: 16384
Resumed vs fresh difference: 1.426e-04


### Step 5: What If You Tighten the Tolerance More Than Once?

The benefit of resumption is clearer when you solve a whole sequence of tighter tolerances. The fresh workflow repeats work at every tolerance, while the resumed workflow only adds the extra work needed for the next target.


In [6]:
tols = [1e-3, 5e-4, 2.5e-4, 1.25e-4]

function compare_tolerance_sequence(tols)
    fresh_rows = NamedTuple[]
    for eps in tols
        r = integrate(make_solver(eps))
        push!(fresh_rows, (abs_tol=eps, n_total=r.data[:n_total], time=r.data[:time_integrate], solution=r.solution))
    end

    resume_rows = NamedTuple[]
    resume_state = nothing
    for eps in tols
        r = isnothing(resume_state) ? integrate(make_solver(eps)) : integrate(make_solver(eps); resume=resume_state)
        push!(resume_rows, (abs_tol=eps, n_total=r.data[:n_total], time=r.data[:time_integrate], solution=r.solution))
        resume_state = r.data
    end
    return fresh_rows, resume_rows
end

fresh_rows, resume_rows = compare_tolerance_sequence(tols)
println("Fresh solves:")
foreach(println, fresh_rows)
println("
Resumed solves:")
foreach(println, resume_rows)

fresh_total_samples = sum(row.n_total for row in fresh_rows)
resume_total_samples = resume_rows[end].n_total
println("
Total fresh samples across all solves: ", fresh_total_samples)
println("Samples after chained resumption: ", resume_total_samples)

@assert resume_total_samples ≤ fresh_total_samples
@assert all(abs(f.solution - r.solution) ≤ f.abs_tol for (f, r) in zip(fresh_rows, resume_rows))


Fresh solves:
(abs_tol

 = 0.001, n_total = 16384, time = 0.0003199577331542969, solution = -0.062223849929872836)
(abs_tol = 0.0005, n_total = 16384, time = 0.00034308433532714844, solution = -0.062223849929872836)
(abs_tol = 0.00025, n_total = 32768, time = 0.004494190216064453, solution = -0.06222641556727295)
(abs_tol = 0.000125, n_total = 65536, time = 0.0018930435180664062, solution = -0.062378451918581844)

Resumed solves:
(abs_tol = 0.001, n_total = 16384, time = 0.0002579689025878906, solution = -0.062223849929872836)
(abs_tol = 0.0005, n_total = 32768, time = 0.0008769035339355469, solution = -0.06236903057758148)
(abs_tol = 0.00025, n_total = 65536, time = 0.0036039352416992188, solution = -0.06240973714001785)
(abs_tol = 0.000125, n_total = 131072, time = 0.005567073822021484, solution = -0.06238140666549654)

Total fresh samples across all solves: 131072
Samples after chained resumption: 131072
